In [3]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "208"  # Station ID for Laubenheim
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists

    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MjA4JEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "208",
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/laubenheim_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()



No data found for 2012-01

No data found for 2012-02

No data found for 2012-03

No data found for 2012-04

No data found for 2012-05

No data found for 2012-06

Data saved to: data/laubenheim_2012_07.csv

Data saved to: data/laubenheim_2012_08.csv

Data saved to: data/laubenheim_2012_09.csv

Data saved to: data/laubenheim_2012_10.csv

Data saved to: data/laubenheim_2012_11.csv

Data saved to: data/laubenheim_2012_12.csv

Data saved to: data/laubenheim_2013_01.csv

Data saved to: data/laubenheim_2013_02.csv

Data saved to: data/laubenheim_2013_03.csv

Data saved to: data/laubenheim_2013_04.csv

Data saved to: data/laubenheim_2013_05.csv

Data saved to: data/laubenheim_2013_06.csv

Data saved to: data/laubenheim_2013_07.csv

Data saved to: data/laubenheim_2013_08.csv

Data saved to: data/laubenheim_2013_09.csv

Data saved to: data/laubenheim_2013_10.csv

Data saved to: data/laubenheim_2013_11.csv

Data saved to: data/laubenheim_2013_12.csv

Data saved to: data/laubenheim_2014_01.csv

D

In [4]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "220"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MjIwJEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "220",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_220_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()


































No data found for 2012-01






























No data found for 2012-02
































No data found for 2012-03































No data found for 2012-04
































No data found for 2012-05































No data found for 2012-06
































No data found for 2012-07
































No data found for 2012-08































No data found for 2012-09
































Data saved to: data/station_220_2012_10.csv































Data saved to: data/station_220_2012_11.csv
































Data saved to: data/station_220_2012_12.csv
































Data saved to: data/station_220_2013_01.csv





























Data saved to: data/station_220_2013_02.csv
































Data saved to: data/station_220_2013_03.csv

































/var/folders/vr/j882mf891_d5jrypcnk4y3340000gn/T/ipykernel_5020/3140285239.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')










Data saved to: data/station_220_2017_01.csv





























Data saved to: data/station_220_2017_02.csv
































Data saved to: data/station_220_2017_03.csv































Data saved to: data/station_220_2017_04.csv
































Data saved to: data/station_220_2017_05.csv































Data saved to: data/station_220_2017_06.csv
































Data saved to: data/station_220_2017_07.csv
































Data saved to: data/station_220_2017_08.csv































Data saved to: data/station_220_2017_09.csv
































Data saved to: data/station_220_2017_10.csv































Data saved to: data/station_220_2017_11.csv
































Data saved to: data/station_220_2017_12.csv
































Data saved to: data/station_220_2018_01.csv





























Data saved to:

/var/folders/vr/j882mf891_d5jrypcnk4y3340000gn/T/ipykernel_5020/3140285239.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')




















Data saved to: data/station_220_2023_08.csv































Data saved to: data/station_220_2023_09.csv
































Data saved to: data/station_220_2023_10.csv































Data saved to: data/station_220_2023_11.csv
































No data found for 2023-12
































Data saved to: data/station_220_2024_01.csv






























Data saved to: data/station_220_2024_02.csv
































Data saved to: data/station_220_2024_03.csv































Data saved to: data/station_220_2024_04.csv
































Data saved to: data/station_220_2024_05.csv































Data saved to: data/station_220_2024_06.csv
































Data saved to: data/station_220_2024_07.csv
































Data saved to: data/station_220_2024_08.csv































Data saved to: data

In [5]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "131"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MTMxJEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "131",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_131_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()


































Data saved to: data/station_131_2012_01.csv






























Data saved to: data/station_131_2012_02.csv
































Data saved to: data/station_131_2012_03.csv































Data saved to: data/station_131_2012_04.csv
































Data saved to: data/station_131_2012_05.csv































Data saved to: data/station_131_2012_06.csv
































Data saved to: data/station_131_2012_07.csv
































Data saved to: data/station_131_2012_08.csv































Data saved to: data/station_131_2012_09.csv
































Data saved to: data/station_131_2012_10.csv































Data saved to: data/station_131_2012_11.csv
































Data saved to: data/station_131_2012_12.csv
































Data saved to: data/station_131_2013_01.csv




















In [6]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "063"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MDYzJEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "063",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_063_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()


































Data saved to: data/station_063_2012_01.csv






























Data saved to: data/station_063_2012_02.csv
































Data saved to: data/station_063_2012_03.csv































Data saved to: data/station_063_2012_04.csv
































Data saved to: data/station_063_2012_05.csv































Data saved to: data/station_063_2012_06.csv
































Data saved to: data/station_063_2012_07.csv
































Data saved to: data/station_063_2012_08.csv































Data saved to: data/station_063_2012_09.csv
































Data saved to: data/station_063_2012_10.csv































Data saved to: data/station_063_2012_11.csv
































Data saved to: data/station_063_2012_12.csv
































Data saved to: data/station_063_2013_01.csv




















In [7]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "244"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MjQ0JEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "244",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_244_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()


































No data found for 2012-01






























No data found for 2012-02
































No data found for 2012-03































No data found for 2012-04
































No data found for 2012-05































No data found for 2012-06
































No data found for 2012-07
































No data found for 2012-08































No data found for 2012-09
































No data found for 2012-10































No data found for 2012-11
































No data found for 2012-12
































No data found for 2013-01





























No data found for 2013-02
































No data found for 2013-03































No data found for 2013-04
































No data found for 2013-05


























/var/folders/vr/j882mf891_d5jrypcnk4y3340000gn/T/ipykernel_5020/3836147787.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')



















Data saved to: data/station_244_2015_11.csv
































Data saved to: data/station_244_2015_12.csv
































Data saved to: data/station_244_2016_01.csv






























Data saved to: data/station_244_2016_02.csv
































Data saved to: data/station_244_2016_03.csv































Data saved to: data/station_244_2016_04.csv
































Data saved to: data/station_244_2016_05.csv































Data saved to: data/station_244_2016_06.csv
































Data saved to: data/station_244_2016_07.csv
































Data saved to: data/station_244_2016_08.csv































Data saved to: data/station_244_2016_09.csv
































Data saved to: data/station_244_2016_10.csv































Data saved to: data/station_244_2016_11.csv
































D

In [9]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "129"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MTI5JEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "129",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_129_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()


































Data saved to: data/station_129_2012_01.csv






























Data saved to: data/station_129_2012_02.csv
































Data saved to: data/station_129_2012_03.csv































Data saved to: data/station_129_2012_04.csv
































Data saved to: data/station_129_2012_05.csv































Data saved to: data/station_129_2012_06.csv
































Data saved to: data/station_129_2012_07.csv
































Data saved to: data/station_129_2012_08.csv































Data saved to: data/station_129_2012_09.csv
































Data saved to: data/station_129_2012_10.csv































Data saved to: data/station_129_2012_11.csv
































Data saved to: data/station_129_2012_12.csv
































Data saved to: data/station_129_2013_01.csv




















In [10]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "261"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MjYxJEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "261",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_261_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()


































No data found for 2012-01






























No data found for 2012-02
































No data found for 2012-03































No data found for 2012-04
































No data found for 2012-05































No data found for 2012-06
































No data found for 2012-07
































No data found for 2012-08































No data found for 2012-09
































No data found for 2012-10































No data found for 2012-11
































No data found for 2012-12
































No data found for 2013-01





























No data found for 2013-02
































No data found for 2013-03































No data found for 2013-04
































No data found for 2013-05


























/var/folders/vr/j882mf891_d5jrypcnk4y3340000gn/T/ipykernel_5020/846487907.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')











Data saved to: data/station_261_2019_01.csv





























Data saved to: data/station_261_2019_02.csv
































Data saved to: data/station_261_2019_03.csv































Data saved to: data/station_261_2019_04.csv
































Data saved to: data/station_261_2019_05.csv































Data saved to: data/station_261_2019_06.csv
































Data saved to: data/station_261_2019_07.csv
































Data saved to: data/station_261_2019_08.csv































Data saved to: data/station_261_2019_09.csv
































Data saved to: data/station_261_2019_10.csv































Data saved to: data/station_261_2019_11.csv
































Data saved to: data/station_261_2019_12.csv
































Data saved to: data/station_261_2020_01.csv






























Data saved t

/var/folders/vr/j882mf891_d5jrypcnk4y3340000gn/T/ipykernel_5020/846487907.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')





























Data saved to: data/station_261_2020_10.csv































Data saved to: data/station_261_2020_11.csv
































Data saved to: data/station_261_2020_12.csv
































Data saved to: data/station_261_2021_01.csv





























Data saved to: data/station_261_2021_02.csv
































Data saved to: data/station_261_2021_03.csv































Data saved to: data/station_261_2021_04.csv
































Data saved to: data/station_261_2021_05.csv































Data saved to: data/station_261_2021_06.csv
































Data saved to: data/station_261_2021_07.csv
































Data saved to: data/station_261_2021_08.csv































Data saved to: data/station_261_2021_09.csv
































Data saved to: data/station_261_2021_10.csv


























/var/folders/vr/j882mf891_d5jrypcnk4y3340000gn/T/ipykernel_5020/846487907.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')




























Data saved to: data/station_261_2024_08.csv































Data saved to: data/station_261_2024_09.csv
































Data saved to: data/station_261_2024_10.csv































Data saved to: data/station_261_2024_11.csv
































Data saved to: data/station_261_2024_12.csv


In [11]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "265"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MjY1JEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "265",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_265_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()


































No data found for 2012-01






























No data found for 2012-02
































No data found for 2012-03































No data found for 2012-04
































No data found for 2012-05































No data found for 2012-06
































No data found for 2012-07
































No data found for 2012-08































No data found for 2012-09
































No data found for 2012-10































No data found for 2012-11
































No data found for 2012-12
































No data found for 2013-01





























No data found for 2013-02
































No data found for 2013-03































No data found for 2013-04
































No data found for 2013-05


























/var/folders/vr/j882mf891_d5jrypcnk4y3340000gn/T/ipykernel_5020/112253843.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')





























Data saved to: data/station_265_2020_10.csv































Data saved to: data/station_265_2020_11.csv
































Data saved to: data/station_265_2020_12.csv
































Data saved to: data/station_265_2021_01.csv





























Data saved to: data/station_265_2021_02.csv
































Data saved to: data/station_265_2021_03.csv































Data saved to: data/station_265_2021_04.csv
































Data saved to: data/station_265_2021_05.csv































Data saved to: data/station_265_2021_06.csv
































Data saved to: data/station_265_2021_07.csv
































Data saved to: data/station_265_2021_08.csv































Data saved to: data/station_265_2021_09.csv
































Data saved to: data/station_265_2021_10.csv


























In [12]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "097"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MDk3JEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "097",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_097_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()

































Data saved to: data/station_097_2012_01.csv






























Data saved to: data/station_097_2012_02.csv
































Data saved to: data/station_097_2012_03.csv































Data saved to: data/station_097_2012_04.csv
































Data saved to: data/station_097_2012_05.csv































Data saved to: data/station_097_2012_06.csv
































Data saved to: data/station_097_2012_07.csv
































Data saved to: data/station_097_2012_08.csv































Data saved to: data/station_097_2012_09.csv
































Data saved to: data/station_097_2012_10.csv































Data saved to: data/station_097_2012_11.csv
































Data saved to: data/station_097_2012_12.csv
































Data saved to: data/station_097_2013_01.csv




















/var/folders/vr/j882mf891_d5jrypcnk4y3340000gn/T/ipykernel_5020/619010303.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')



























Data saved to: data/station_097_2018_04.csv
































Data saved to: data/station_097_2018_05.csv































Data saved to: data/station_097_2018_06.csv
































Data saved to: data/station_097_2018_07.csv
































Data saved to: data/station_097_2018_08.csv































Data saved to: data/station_097_2018_09.csv
































Data saved to: data/station_097_2018_10.csv































Data saved to: data/station_097_2018_11.csv
































Data saved to: data/station_097_2018_12.csv
































Data saved to: data/station_097_2019_01.csv





























Data saved to: data/station_097_2019_02.csv
































Data saved to: data/station_097_2019_03.csv































Data saved to: data/station_097_2019_04.csv




























In [13]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "228"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MjI4JEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "228",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_228_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()


































No data found for 2012-01






























No data found for 2012-02
































No data found for 2012-03































No data found for 2012-04
































No data found for 2012-05































No data found for 2012-06
































No data found for 2012-07
































No data found for 2012-08































No data found for 2012-09
































No data found for 2012-10































No data found for 2012-11
































No data found for 2012-12
































Data saved to: data/station_228_2013_01.csv





























Data saved to: data/station_228_2013_02.csv
































Data saved to: data/station_228_2013_03.csv































Data saved to: data/station_228_2013_04.csv












In [14]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import time
import os
import base64

class DFLDDataCollector:
    def __init__(self):
        self.base_url = "https://www.dfld.de/Mess/StatTag.php"
        self.station_id = "142"  # Updated Station ID
        self.region = "001"      # Region code
        os.makedirs('data', exist_ok=True)  # Ensure data directory exists
        
    def generate_aio_parameter(self, date_str):
        """Generate the AiO parameter required for the API request"""
        encoded_date = base64.b64encode(date_str.encode()).decode()
        pattern = f"RG89RCRATD1HJEBSPTEkQFM9MTQyJEBLPTAkQEQ9{encoded_date}JEBTVD01JEBaPTAwOjAwOjAwJEBaVD0kQEE9MSRAUHM9JEBNTnI9MCRATm9PcHQ9MA!"
        return pattern

    def parse_data(self, raw_data, date):
        """Parse the raw data into a structured format"""
        try:
            lines = raw_data.split('\n')
            data_start = -1
            station_name = None
            
            for i, line in enumerate(lines):
                if line.startswith('Zeit;'):
                    data_start = i
                    break
                elif "(" in line and ")" in line:
                    station_name = line.split('(')[0].strip()
            
            if data_start == -1:
                return None

            header = lines[data_start].split(';')
            data_lines = [line.split(';') for line in lines[data_start+1:] if line.strip() and ';' in line]
            
            df = pd.DataFrame(data_lines, columns=header)
            df.columns = ['time', 'db_a']  # Standard column names
            
            df['date'] = date.strftime('%Y-%m-%d')
            df['station_name'] = station_name
            df['db_a'] = pd.to_numeric(df['db_a'], errors='coerce')
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
            
            df = df.dropna().drop_duplicates()  # Ensure no duplicate or invalid rows
            return df
        except Exception as e:
            print(f"Error parsing data for {date}: {e}")
            return None

    def get_daily_data(self, date):
        """Fetch data for a specific date"""
        date_str = date.strftime("%d.%m.%Y")
        aio_param = self.generate_aio_parameter(date_str)
        params = {
            "R": "001",
            "S": "142",  # Updated Station ID
            "D": date_str,
            "AS": "269557505",
            "AiO": aio_param
        }
        try:
            print(f"\nCollecting data for {date_str}...")
            response = requests.get(self.base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
            
            os.makedirs('debug', exist_ok=True)
            with open(f'debug/response_{date_str.replace(".", "_")}.html', 'w', encoding='utf-8') as f:
                f.write(response.text)
            
            if response.status_code == 200:
                return self.parse_data(response.text, date)
            else:
                print(f"Error fetching data for {date_str}: Status code {response.status_code}")
                return None
        except Exception as e:
            print(f"Exception occurred while fetching data for {date_str}: {e}")
            return None

    def collect_monthly_data(self, year, month):
        """Collect data for an entire month"""
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) if month < 12 else datetime(year + 1, 1, 1)
        
        all_data = []
        current_date = start_date
        
        while current_date < end_date:
            daily_data = self.get_daily_data(current_date)
            if daily_data is not None and not daily_data.empty:
                all_data.append(daily_data)
            current_date += timedelta(days=1)
            time.sleep(1)  # Avoid overwhelming the server
        
        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame(columns=['time', 'db_a', 'date', 'station_name', 'datetime'])

def main():
    collector = DFLDDataCollector()
    
    for year in range(2012, 2025):  # Loop from 2012 to 2024
        for month in range(1, 13):  # Loop through all months
            print(f"\nCollecting data for {year}-{month:02d}...")
            monthly_data = collector.collect_monthly_data(year, month)
            
            if not monthly_data.empty:
                output_file = f'data/station_142_{year}_{month:02d}.csv'
                monthly_data.to_csv(output_file, index=False)
                print(f"Data saved to: {output_file}")
            else:
                print(f"No data found for {year}-{month:02d}")

if __name__ == "__main__":
    main()


































Data saved to: data/station_142_2012_01.csv






























Data saved to: data/station_142_2012_02.csv
































Data saved to: data/station_142_2012_03.csv































Data saved to: data/station_142_2012_04.csv
































Data saved to: data/station_142_2012_05.csv































Data saved to: data/station_142_2012_06.csv
































Data saved to: data/station_142_2012_07.csv
































Data saved to: data/station_142_2012_08.csv































Data saved to: data/station_142_2012_09.csv
































Data saved to: data/station_142_2012_10.csv































Data saved to: data/station_142_2012_11.csv
































Data saved to: data/station_142_2012_12.csv
































Data saved to: data/station_142_2013_01.csv




















In [16]:
import os
import pandas as pd

def merge_station_data(directory):
    """Merge all CSV files based on station name and date."""
    all_files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    station_data = {}
    
    for file in all_files:
        file_path = os.path.join(directory, file)
        try:
            df = pd.read_csv(file_path)
            if 'station_name' in df.columns and 'date' in df.columns:
                station_name = df['station_name'].iloc[0]
                if station_name not in station_data:
                    station_data[station_name] = []
                station_data[station_name].append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    merged_data = {}
    for station, dfs in station_data.items():
        merged_df = pd.concat(dfs, ignore_index=True).drop_duplicates()
        merged_data[station] = merged_df
        output_file = os.path.join(directory, f"merged_{station.replace(' ', '_')}.csv")
        merged_df.to_csv(output_file, index=False)
        print(f"Merged data saved to: {output_file}")
    
    return merged_data

# Run the merging function
merged_datasets = merge_station_data("data")


Merged data saved to: data/merged_Mainz/Oberstadt_ooo.csv
Merged data saved to: data/merged_Mainz/Laubenheim_ooo.csv
Merged data saved to: data/merged_Mainz/Weisenau_2_ooo.csv
Merged data saved to: data/merged_Mainz/Laubenheim_2.csv
Merged data saved to: data/merged_Mainz/Hechtsheim_1_ooo.csv
Merged data saved to: data/merged_Mainz/Universit�tsmedizin_ooo.csv
Merged data saved to: data/merged_Mainz/Lerchenberg.csv
Merged data saved to: data/merged_Mainz/Bretzenheim_ooo.csv
Merged data saved to: data/merged_Mainz/Lerchenberg_ooo.csv
Merged data saved to: data/merged_Mainz/Ebersheim.csv
Merged data saved to: data/merged_Mainz/Hechtsheim_2.csv
